# 03_axis1_axis4
Axis 1 (post-hoc realisability) and Axis 4 (empirical boundary calibration).
Axis 1 asks whether the direction a recourse prescribes actually occurs in the
population: we observe realised (t -> t+1) changes in the actionable features
among at-risk persons and estimate the quantile of change that recourse
magnitudes correspond to. Axis 4 replaces arbitrary feasibility bounds with
empirically observed change distributions, reported unconditionally and
conditional on age band. Greyscale figures + CSV tables are written to results/.

In [1]:
%run 00_config.ipynb

PROJ_DIR: /home/claude/recourse_khp
1y pairs: [(2019, 2020), (2020, 2021), (2021, 2022), (2022, 2023), (2023, 2024)]
2y pairs: [(2019, 2021), (2020, 2022), (2021, 2023), (2022, 2024)]
registry loaded
helpers loaded
00_config ready


In [2]:
# --- Load transitions and the prescribed recourse deltas ---
tr=pd.read_parquet(os.path.join(DATA_DIR,"transitions_1y.parquet"))
tr2=pd.read_parquet(os.path.join(DATA_DIR,"transitions_2y.parquet"))
rec=pd.read_parquet(os.path.join(DATA_DIR,"recourse_declined.parquet"))
print("1y transitions:",tr.shape," recourse cases:",rec.shape)

# realised change in actionable features among adults, t0->t1
for a in ["BMI","PA_WALK","ALC_FREQ"]:
    tr[f"realised_d_{a}"]=tr[f"{a}_t1"]-tr[f"{a}_t0"]
# focus on HTN at-risk population (disease-free at t0) for policy relevance
atr=tr[tr["HTN_atrisk"]==1].copy()
print("HTN at-risk person-pairs:",len(atr))

1y transitions: (49284, 50)  recourse cases: (540, 13)
HTN at-risk person-pairs: 30853


In [3]:
# --- AXIS 1: realised change distributions vs prescribed recourse magnitude ---
# The recourse most often prescribes BMI reduction; quantify how often the
# population realises a reduction of the prescribed size within one year.
presc_bmi=-rec["d_BMI"]                       # prescribed reduction magnitude (>=0)
presc_bmi=presc_bmi[presc_bmi>0]
realised_red=-atr["realised_d_BMI"].dropna()  # realised reduction (>0 = lost BMI)

# For each prescribed magnitude, empirical achievement rate = P(realised reduction >= prescribed)
qs=[0.5,0.75,0.9]
rows=[]
for q in qs:
    m=presc_bmi.quantile(q)
    ach=(realised_red>=m).mean()
    rows.append({"prescribed_BMI_reduction_quantile":q,
                 "prescribed_reduction":round(float(m),2),
                 "pop_achievement_rate":round(float(ach),3)})
ax1=pd.DataFrame(rows); savetable(ax1,"t03_axis1_achievement", index=False)
print(ax1.to_string(index=False))

# realised BMI change achieved-quantile for the *median* prescribed reduction
med=presc_bmi.median()
emp_q=(realised_red<med).mean()
print(f"\nMedian prescribed BMI reduction = {med:.2f}; "
      f"it sits at the {100*emp_q:.1f}th percentile of realised reductions "
      f"(=> only {100*(1-emp_q):.1f}% of at-risk adults realise >= that).")

saved: t03_axis1_achievement.csv
 prescribed_BMI_reduction_quantile  prescribed_reduction  pop_achievement_rate
                              0.50                  4.34                 0.005
                              0.75                  6.82                 0.001
                              0.90                  9.34                 0.000

Median prescribed BMI reduction = 4.34; it sits at the 99.5th percentile of realised reductions (=> only 0.5% of at-risk adults realise >= that).


In [4]:
# --- AXIS 1 figure: realised 1-year BMI change with prescribed reduction markers ---
fig,ax=plt.subplots(figsize=(6.0,4.0))
vals=atr["realised_d_BMI"].dropna()
vals=vals[vals.between(-10,10)]
sns.histplot(vals,bins=60,color="#666666",edgecolor="white",linewidth=0.3,ax=ax,stat="density")
for q,ls in zip([0.5,0.75,0.9],["-","--",":"]):
    m=-presc_bmi.quantile(q)     # prescribed change is negative (reduction)
    ax.axvline(m,color="#000000",lw=1.0,ls=ls,label=f"prescribed q{int(q*100)} = {m:.1f}")
ax.axvline(0,color="#999999",lw=0.8)
ax.set_xlabel("Realised 1-year change in BMI"); ax.set_ylabel("Density")
ax.legend(frameon=False, fontsize=8)
savefig(fig,"f03_axis1_bmi_realised"); plt.close(fig)
print("axis-1 figure saved")

saved: f03_axis1_bmi_realised.png / f03_axis1_bmi_realised.pdf
axis-1 figure saved


In [5]:
# --- AXIS 4: empirical feasibility bounds, unconditional and age-conditional ---
# Bound = the realised reduction achieved by a given upper quantile of the population.
def bounds_by_group(df, feature, grp=None, qs=(0.5,0.75,0.9,0.95)):
    red=-(df[f"{feature}_t1"]-df[f"{feature}_t0"])   # reduction magnitude
    d=pd.DataFrame({"red":red, "g": (df[grp] if grp else "ALL")}).dropna()
    d=d[d["red"].between(-15,15)]
    out=[]
    for g,gg in d.groupby("g", observed=True):
        row={"group":g,"n":len(gg)}
        for q in qs:
            row[f"bound_q{int(q*100)}"]=round(float(gg["red"].quantile(q)),2)
        out.append(row)
    return pd.DataFrame(out)

uncond=bounds_by_group(atr,"BMI"); uncond.insert(0,"scope","unconditional")
byage =bounds_by_group(atr,"BMI",grp="AGEG_t0"); byage.insert(0,"scope","by_age")
ax4=pd.concat([uncond,byage],ignore_index=True)
savetable(ax4,"t03_axis4_bounds", index=False)
print(ax4.to_string(index=False))

saved: t03_axis4_bounds.csv
        scope group     n  bound_q50  bound_q75  bound_q90  bound_q95
unconditional   ALL 29700       -0.0       0.37       1.06       1.73
       by_age 19-29  2558       -0.0       0.40       1.34       2.03
       by_age 30-39  4020       -0.0       0.38       1.31       2.00
       by_age 40-49  5814       -0.0       0.35       1.01       1.65
       by_age 50-59  5783        0.0       0.35       0.98       1.58
       by_age 60-69  6343        0.0       0.35       0.91       1.52
       by_age   70+  5182       -0.0       0.39       1.10       1.78


In [6]:
# --- AXIS 4 figure: age-conditional empirical feasibility bound (BMI reduction) ---
order=["19-29","30-39","40-49","50-59","60-69","70+"]
bg=byage[byage["group"].isin(order)].set_index("group").reindex(order)
fig,ax=plt.subplots(figsize=(6.2,4.0))
xs=np.arange(len(order)); w=0.25
for k,(q,hatch) in enumerate(zip(["bound_q75","bound_q90","bound_q95"],["","//","xx"])):
    ax.bar(xs+(k-1)*w, bg[q].values, width=w, label=q.replace("bound_",""),
           color="#888888", edgecolor="black", hatch=hatch, linewidth=0.5)
ax.set_xticks(xs); ax.set_xticklabels(order)
ax.set_xlabel("Age band"); ax.set_ylabel("Empirical BMI-reduction bound (1 year)")
ax.legend(frameon=False, title=None, fontsize=8)
savefig(fig,"f03_axis4_agebounds"); plt.close(fig)
print("axis-4 figure saved")

saved: f03_axis4_agebounds.png / f03_axis4_agebounds.pdf
axis-4 figure saved


In [7]:
# --- Feasibility labelling: is each declined person's recourse within the
# age-conditional empirical bound? (used by axes 2-3 as feasible vs infeasible) ---
bound90=byage.set_index("group")["bound_q90"].to_dict()
# attach age band to recourse cases via the modelling frame order is lost;
# recompute using panel baseline (first obs per person) to get age band
panel=pd.read_parquet(os.path.join(DATA_DIR,"panel_long.parquet"))
base=panel.sort_values("year").groupby(KEY,as_index=False).first()
# The recourse table lacks PIDWON; re-derive by matching baseline actionable rows.
# For robustness we instead label feasibility purely by prescribed magnitude vs
# the *population* 90th-percentile bound (age-agnostic) and by age via merge in 04.
pop_b90=float((-atr["realised_d_BMI"]).quantile(0.90))
rec["feasible_pop"]=(-rec["d_BMI"] <= pop_b90).astype(int)
rec.to_parquet(os.path.join(DATA_DIR,"recourse_feasibility.parquet"), index=False)
print(f"population 90th-pct BMI-reduction bound = {pop_b90:.2f}")
print("feasible under population bound:",
      f'{100*rec["feasible_pop"].mean():.1f}% of recourse cases')

population 90th-pct BMI-reduction bound = 1.06
feasible under population bound: 13.5% of recourse cases
